In [321]:
import pandas as pd
import os
import requests
import time

In [322]:
BASE_URL = "https://fantasy.premierleague.com/api/"

In [323]:
response = requests.get(BASE_URL + "bootstrap-static/")
data = response.json()

In [324]:
# CHECKING WHAT DATA RETURNS
data.keys()

dict_keys(['chips', 'events', 'game_settings', 'game_config', 'phases', 'teams', 'total_players', 'element_stats', 'element_types', 'elements'])

In [325]:
# CHECKING ELEMENTS OUTPUT
print(data["elements"][0].keys())

dict_keys(['can_transact', 'can_select', 'chance_of_playing_next_round', 'chance_of_playing_this_round', 'code', 'cost_change_event', 'cost_change_event_fall', 'cost_change_start', 'cost_change_start_fall', 'price_change_percent', 'dreamteam_count', 'element_type', 'ep_next', 'ep_this', 'event_points', 'first_name', 'form', 'id', 'in_dreamteam', 'news', 'news_added', 'now_cost', 'photo', 'points_per_game', 'removed', 'second_name', 'selected_by_percent', 'special', 'squad_number', 'status', 'team', 'team_code', 'total_points', 'transfers_in', 'transfers_in_event', 'transfers_out', 'transfers_out_event', 'value_form', 'value_season', 'web_name', 'known_name', 'region', 'team_join_date', 'birth_date', 'has_temporary_code', 'opta_code', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_intercept

In [326]:
# CHECKING WHAT EVENT ENDPOINT RETURNS
data["events"][0]

{'id': 1,
 'name': 'Gameweek 1',
 'deadline_time': '2025-08-15T17:30:00Z',
 'release_time': None,
 'average_entry_score': 54,
 'finished': True,
 'data_checked': True,
 'highest_scoring_entry': 3772644,
 'deadline_time_epoch': 1755279000,
 'deadline_time_game_offset': 0,
 'highest_score': 127,
 'is_previous': False,
 'is_current': False,
 'is_next': False,
 'cup_leagues_created': False,
 'h2h_ko_matches_created': False,
 'can_enter': False,
 'can_manage': False,
 'released': True,
 'ranked_count': 9469118,
 'overrides': {'rules': {},
  'scoring': {},
  'element_types': [],
  'pick_multiplier': None},
 'chip_plays': [{'chip_name': 'bboost', 'num_played': 342779},
  {'chip_name': '3xc', 'num_played': 272642}],
 'most_selected': 235,
 'most_transferred_in': 1,
 'top_element': 531,
 'top_element_info': {'id': 531, 'points': 17},
 'transfers_made': 0,
 'most_captained': 381,
 'most_vice_captained': 235}

In [327]:
test = requests.get(BASE_URL + "element-summary/1/").json()
print(test["history"][0].keys())

dict_keys(['element', 'fixture', 'opponent_team', 'total_points', 'was_home', 'kickoff_time', 'team_h_score', 'team_a_score', 'round', 'modified', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries', 'tackles', 'defensive_contribution', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'value', 'transfers_balance', 'selected', 'transfers_in', 'transfers_out'])


In [328]:
current_gw = None
for event in data["events"]:
    if event["is_current"]:
        current_gw = event["id"]
        break

user_input = input(f"Press Enter for current GW ({current_gw}) or type a GW number: ").strip()
gw = int(user_input) if user_input else current_gw
print(f"Downloading GW{gw}...")

# ── SECTION 3 ── Players + Teams from bootstrap
# NOTE: using original API field names (code, element_type) to match Power BI expectations
players = {}
for player in data["elements"]:
    players[player["id"]] = {
        "name": f"{player['first_name']} {player['second_name']}",
        "web_name": player["web_name"],
        "team_id": player["team"],
        "element_type": player["element_type"],
        "code": player["code"],
        "now_cost": player["now_cost"] / 10,
        "status": player.get("status"),
        "news": player.get("news"),
        "corners_and_indirect_freekicks_order": player.get("corners_and_indirect_freekicks_order"),
        "direct_freekicks_order": player.get("direct_freekicks_order"),
        "penalties_order": player.get("penalties_order"),
        "selected_by_percent": player.get("selected_by_percent")
    }

teams = {}
for team in data["teams"]:
    teams[team["id"]] = team["name"]

# ── SECTION 4 ── Fixtures + Live GW data
response = requests.get(BASE_URL + "fixtures/")
fixtures_data = response.json()

response1 = requests.get(BASE_URL + f"event/{gw}/live/")
live_data = response1.json()

# Filter fixtures to this gameweek, keyed by fixture ID
fixtures = {}
for fixture in fixtures_data:
    if fixture["event"] == gw:
        fixtures[fixture["id"]] = {
            "home_team_id": fixture["team_h"],
            "away_team_id": fixture["team_a"],
            "home_score": fixture["team_h_score"],
            "away_score": fixture["team_a_score"],
            "kickoff_time": fixture["kickoff_time"]
        }

# ── SECTION 5 ── Helper function to build a row
# NOTE: output columns use original names (code, element_type) to match Power BI
def build_row(player_id, player, gw, fixture_id, fixtures, stats, in_dreamteam=None):
    opponent_id = None
    team_score = None
    opponent_score = None
    kickoff_time = None
    h_a = None

    if fixture_id and fixture_id in fixtures:
        fixture = fixtures[fixture_id]
        kickoff_time = fixture["kickoff_time"]

        if player["team_id"] == fixture["home_team_id"]:
            opponent_id = fixture["away_team_id"]
            team_score = fixture["home_score"]
            opponent_score = fixture["away_score"]
            h_a = "Home"
        else:
            opponent_id = fixture["home_team_id"]
            team_score = fixture["away_score"]
            opponent_score = fixture["home_score"]
            h_a = "Away"

    return {
        "player_id": player_id,
        "code": player.get("code"),
        "gameweek": gw,
        "fixture_id": fixture_id,
        "name": player.get("name"),
        "web_name": player.get("web_name"),
        "element_type": player.get("element_type"),
        "selected_by_percent": player.get("selected_by_percent"),
        "value": player.get("now_cost"),
        "team_id": player.get("team_id"),
        "opponent_id": opponent_id,
        "h_a": h_a,
        "kickoff_time": kickoff_time,
        "team_score": team_score,
        "opponent_score": opponent_score,
        "minutes": stats.get("minutes", 0),
        "goals_scored": stats.get("goals_scored", 0),
        "assists": stats.get("assists", 0),
        "clean_sheets": stats.get("clean_sheets", 0),
        "goals_conceded": stats.get("goals_conceded", 0),
        "yellow_cards": stats.get("yellow_cards", 0),
        "red_cards": stats.get("red_cards", 0),
        "bonus": stats.get("bonus", 0),
        "bps": stats.get("bps", 0),
        "total_points": stats.get("total_points", 0),
        "xG": stats.get("expected_goals", 0),
        "xA": stats.get("expected_assists", 0),
        "xGI": stats.get("expected_goal_involvements", 0),
        "xGC": stats.get("expected_goals_conceded", 0),
        "DefCon": stats.get("defensive_contribution", 0),
        "ict_index": stats.get("ict_index", 0),
        "influence": stats.get("influence", 0),
        "creativity": stats.get("creativity", 0),
        "threat": stats.get("threat", 0),
        "in_dreamteam": in_dreamteam,
        "status": player.get("status"),
        "news": player.get("news"),
        "corners_indirect_freekicks_order": player.get("corners_and_indirect_freekicks_order"),
        "direct_freekicks_order": player.get("direct_freekicks_order"),
        "penalties_order": player.get("penalties_order"),
    }

# ── SECTION 6 ── Main loop
rows = []

for element in live_data["elements"]:
    player_id = element["id"]
    stats = element["stats"]
    player = players.get(player_id, {})
    explains = element.get("explain", [])

    if len(explains) > 1:
        # ── Double GW ── call element-summary for per-fixture stats
        summary_response = requests.get(BASE_URL + f"element-summary/{player_id}/")
        summary_data = summary_response.json()

        for game in summary_data["history"]:
            if game["round"] == gw:
                fixture_id = game["fixture"]

                per_fixture_stats = {
                    "minutes": game.get("minutes", 0),
                    "goals_scored": game.get("goals_scored", 0),
                    "assists": game.get("assists", 0),
                    "clean_sheets": game.get("clean_sheets", 0),
                    "goals_conceded": game.get("goals_conceded", 0),
                    "yellow_cards": game.get("yellow_cards", 0),
                    "red_cards": game.get("red_cards", 0),
                    "bonus": game.get("bonus", 0),
                    "bps": game.get("bps", 0),
                    "total_points": game.get("total_points", 0),
                    "expected_goals": game.get("expected_goals", 0),
                    "expected_assists": game.get("expected_assists", 0),
                    "expected_goal_involvements": game.get("expected_goal_involvements", 0),
                    "expected_goals_conceded": game.get("expected_goals_conceded", 0),
                    "defensive_contribution": game.get("defensive_contribution", 0),
                    "ict_index": game.get("ict_index", 0),
                    "influence": game.get("influence", 0),
                    "creativity": game.get("creativity", 0),
                    "threat": game.get("threat", 0),
                }

                row = build_row(player_id, player, gw, fixture_id, fixtures, per_fixture_stats, in_dreamteam=None)
                rows.append(row)

    else:
        # ── Single GW ── use live data as normal
        fixture_id = None
        if explains:
            fixture_id = explains[0].get("fixture")

        row = build_row(player_id, player, gw, fixture_id, fixtures, stats, in_dreamteam=stats.get("in_dreamteam", False))
        rows.append(row)

In [329]:
df = pd.DataFrame(rows)
df.head(2)

,player_id,code,gameweek,fixture_id,name,web_name,element_type,selected_by_percent,value,team_id,...,ict_index,influence,creativity,threat,in_dreamteam,status,news,corners_indirect_freekicks_order,direct_freekicks_order,penalties_order
0,1,154561,33,327,David Raya Martín,Raya,1,33.9,6.0,1,...,2.1,21.0,0.0,0.0,False,a,,NaN,NaN,NaN
1,2,109745,33,327,Kepa Arrizabalaga Revuelta,Arrizabalaga,1,0.4,4.0,1,...,0.0,0.0,0.0,0.0,False,a,,NaN,NaN,NaN


In [330]:
# Save GW file
df["selected_by_percent"] = pd.to_numeric(df["selected_by_percent"], errors="coerce")
folder = r"B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload"
filename = os.path.join(folder, f"gw{gw}.csv")
df.to_csv(filename, index=False)
print(f"Saved {len(df)} rows to {filename}")
print(df.dtypes)

Saved 1077 rows to B:\Python_Projects\PowerBI\FPL_PBI\FPL-Analysis\25-26 gws upload\gw33.csv
player_id                             int64
code                                  int64
gameweek                              int64
fixture_id                            int64
name                                 object
web_name                             object
element_type                          int64
selected_by_percent                 float64
value                               float64
team_id                               int64
opponent_id                           int64
h_a                                  object
kickoff_time                         object
team_score                            int64
opponent_score                        int64
minutes                               int64
goals_scored                          int64
assists                               int64
clean_sheets                          int64
goals_conceded                        int64
yellow_cards               